In [24]:
import jieba
import pandas as pd 
from collections import Counter
from pyecharts.charts import Line,Pie,Scatter,Bar,Map,Grid
from pyecharts.charts import WordCloud
from pyecharts import options as opts
from pyecharts.globals import ThemeType
from pyecharts.globals import SymbolType
from pyecharts.commons.utils import JsCode

from pyecharts.globals import CurrentConfig, NotebookType,OnlineHostType
CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_NOTEBOOK

In [25]:
import pandas as pd

file_path = '旅游景点.xlsm'
df = pd.read_excel(file_path)

df.head()


,城市,名称,星级,评分,价格,销量,省/市/区,坐标,简介,是否免费,具体地址
0,上海,上海迪士尼乐园,NaN,0.0,325.0,19459,上海·上海·浦东新区,"121.667917,31.149712",每个女孩都有一场迪士尼梦,False,上海市浦东新区川沙镇黄赵路310号上海迪士尼乐园
1,上海,上海海昌海洋公园,4A,0.0,276.5,19406,上海·上海·浦东新区,"121.915647,30.917713",看珍稀海洋生物 | 玩超刺激娱乐项目,False,上海市浦东新区南汇城银飞路166号
2,上海,上海野生动物园,5A,3.6,116.0,6764,上海·上海·浦东新区,"121.728112,31.059636",全球动物聚集地 | 零距离和动物做朋友,False,上海市浦东新区南六公路178号
3,上海,东方绿舟,4A,3.5,40.0,5353,上海·上海·青浦区,"121.015977,31.107866",全国首屈一指的青少年校外教育营地,False,上海市青浦区沪青平公路6888号
4,上海,东方明珠,5A,3.8,54.0,3966,上海·上海·浦东新区,"121.50626,31.245369",感受云端漫步，品味老上海风情,False,上海市浦东新区世纪大道1号


In [26]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

matplotlib.rc('font', family='SimHei')

# 检查数据的缺失值和数据类型
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2443 entries, 0 to 2442
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   城市      2443 non-null   object 
 1   名称      2443 non-null   object 
 2   星级      913 non-null    object 
 3   评分      2443 non-null   float64
 4   价格      2443 non-null   float64
 5   销量      2443 non-null   int64  
 6   省/市/区   2443 non-null   object 
 7   坐标      2443 non-null   object 
 8   简介      2402 non-null   object 
 9   是否免费    2443 non-null   bool   
 10  具体地址    2440 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(7)
memory usage: 193.4+ KB


In [27]:
#数据预处理
#去重
# 打印去重前的行数
print("去重前行数:", len(df))

# 根据第一列"景点名称"进行去重
df = df.drop_duplicates(subset=['名称'])

# 打印去重后的行数
print("去重后行数:", len(df))

去重前行数: 2443
去重后行数: 2443


In [28]:
# 填充星级的缺失值
most_frequent_star = df['星级'].mode()[0]
df['星级'].fillna(most_frequent_star, inplace=True)

# 填充简介和具体地址的缺失值为"未知"
df['简介'].fillna('未知', inplace=True)
df['具体地址'].fillna('未知', inplace=True)

/var/folders/64/bsj3fv297pddm3dh59r9y_nw0000gn/T/ipykernel_23233/1772742197.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['星级'].fillna(most_frequent_star, inplace=True)
/var/folders/64/bsj3fv297pddm3dh59r9y_nw0000gn/T/ipykernel_23233/1772742197.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alw

In [29]:
# 验证填充后的数据
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2443 entries, 0 to 2442
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   城市      2443 non-null   object 
 1   名称      2443 non-null   object 
 2   星级      2443 non-null   object 
 3   评分      2443 non-null   float64
 4   价格      2443 non-null   float64
 5   销量      2443 non-null   int64  
 6   省/市/区   2443 non-null   object 
 7   坐标      2443 non-null   object 
 8   简介      2443 non-null   object 
 9   是否免费    2443 non-null   bool   
 10  具体地址    2443 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(7)
memory usage: 193.4+ KB


In [30]:
df.loc[df['销量']==0,:].head()

,城市,名称,星级,评分,价格,销量,省/市/区,坐标,简介,是否免费,具体地址
301,台湾,复兴桥风景区,4A,0.0,0.0,0,台湾·桃园,"121.35037883956,24.815040245504",带动了北横碧水青山的另一种欢乐风情,False,桃园县复兴乡中正路15号
302,台湾,长春祠,4A,3.5,0.0,0,台湾·花莲,"121.60445032684,24.162302549393",一道飞瀑分流入溪，山水景色绝佳,True,台湾花莲县
303,台湾,淡水老街,4A,0.0,0.0,0,台湾·淡水,"121.44124069779,25.171297755773",美食汇聚的北台湾老街之一,True,台北县淡水镇中正路
304,台湾,士林官邸,4A,5.0,0.0,0,台湾·台北,"121.53013540214,25.093934468272",林木参天、绿树幽深，极富神秘色彩,True,台北市士林区中山北路五段兴福林路口
305,台湾,北安公园,4A,5.0,0.0,0,台湾·台北,"121.52694244942,25.076889096728",满园新绿，温雅如诗，连绵一片绿意盎然,False,台湾台北市中山北路四段北安路旁


In [31]:
df = df[df['销量']!=0] 
df.shape

(2320, 11)

In [32]:
df.sort_values('销量', ascending=False).head()

,城市,名称,星级,评分,价格,销量,省/市/区,坐标,简介,是否免费,具体地址
0,上海,上海迪士尼乐园,4A,0.0,325.0,19459,上海·上海·浦东新区,"121.667917,31.149712",每个女孩都有一场迪士尼梦,False,上海市浦东新区川沙镇黄赵路310号上海迪士尼乐园
1,上海,上海海昌海洋公园,4A,0.0,276.5,19406,上海·上海·浦东新区,"121.915647,30.917713",看珍稀海洋生物 | 玩超刺激娱乐项目,False,上海市浦东新区南汇城银飞路166号
211,北京,故宫,5A,5.0,58.6,15277,北京·北京·东城区,"116.403347,39.922148",世界五大宫之首，穿越与您近在咫尺,False,北京市东城区景山前街4号
2187,陕西,秦始皇帝陵博物院（兵马俑）,5A,4.4,120.0,12714,陕西·西安·临潼区,"109.266029,34.386024",世界第八大奇迹,False,陕西省西安市临潼县秦始皇陵东1.5公里处
474,四川,成都大熊猫繁育研究基地,4A,4.0,52.0,9731,四川·成都·成华区,"104.152603,30.738951",无关黑与白， 不分胖与瘦， 可爱而又温暖,False,四川省成都市成华区外北熊猫大道1375号


In [33]:
df.isnull().sum()

城市       0
名称       0
星级       0
评分       0
价格       0
销量       0
省/市/区    0
坐标       0
简介       0
是否免费     0
具体地址     0
dtype: int64

## 将星级缺失值用‘未知’填充

In [34]:

df['星级'].fillna('未知', inplace=True)
df.isnull().sum()

/var/folders/64/bsj3fv297pddm3dh59r9y_nw0000gn/T/ipykernel_23233/368977410.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['星级'].fillna('未知', inplace=True)


城市       0
名称       0
星级       0
评分       0
价格       0
销量       0
省/市/区    0
坐标       0
简介       0
是否免费     0
具体地址     0
dtype: int64

#  也可以将所有缺失值都用‘未知’填充

In [35]:
# 也可以将所有缺失值都用‘未知’填充
df.fillna('未知', inplace=True)
df.isnull().sum()

城市       0
名称       0
星级       0
评分       0
价格       0
销量       0
省/市/区    0
坐标       0
简介       0
是否免费     0
具体地址     0
dtype: int64

# 按销量排序

In [36]:

df.sort_values('销量', ascending=False).head()

,城市,名称,星级,评分,价格,销量,省/市/区,坐标,简介,是否免费,具体地址
0,上海,上海迪士尼乐园,4A,0.0,325.0,19459,上海·上海·浦东新区,"121.667917,31.149712",每个女孩都有一场迪士尼梦,False,上海市浦东新区川沙镇黄赵路310号上海迪士尼乐园
1,上海,上海海昌海洋公园,4A,0.0,276.5,19406,上海·上海·浦东新区,"121.915647,30.917713",看珍稀海洋生物 | 玩超刺激娱乐项目,False,上海市浦东新区南汇城银飞路166号
211,北京,故宫,5A,5.0,58.6,15277,北京·北京·东城区,"116.403347,39.922148",世界五大宫之首，穿越与您近在咫尺,False,北京市东城区景山前街4号
2187,陕西,秦始皇帝陵博物院（兵马俑）,5A,4.4,120.0,12714,陕西·西安·临潼区,"109.266029,34.386024",世界第八大奇迹,False,陕西省西安市临潼县秦始皇陵东1.5公里处
474,四川,成都大熊猫繁育研究基地,4A,4.0,52.0,9731,四川·成都·成华区,"104.152603,30.738951",无关黑与白， 不分胖与瘦， 可爱而又温暖,False,四川省成都市成华区外北熊猫大道1375号


# Pyecharts数据可视化
# 销量前20热门景点数据
# 线性渐变

In [37]:

color_js = """new echarts.graphic.LinearGradient(0, 0, 1, 0,
    [{offset: 0, color: '#009ad6'}, {offset: 1, color: '#ed1941'}], false)"""


sort_info = df.sort_values(by='销量', ascending=True)
b1 = (
    Bar()
    .add_xaxis(list(sort_info['名称'])[-20:])
    .add_yaxis('热门景点销量', sort_info['销量'].values.tolist()[-20:],itemstyle_opts=opts.ItemStyleOpts(color=JsCode(color_js)))
    .reversal_axis()
    .set_global_opts(
        title_opts=opts.TitleOpts(title='热门景点销量数据'),
        yaxis_opts=opts.AxisOpts(name='景点名称'),
        xaxis_opts=opts.AxisOpts(name='销量'),
        )
    .set_series_opts(label_opts=opts.LabelOpts(position="right"))

)
# 将图形整体右移
g1 = (
    Grid()
        .add(b1, grid_opts=opts.GridOpts(pos_left='20%', pos_right='5%'))  
)
g1.render_notebook()

# 假期出行数据全国地图分布

In [38]:
dictcode = {'北京': '北京市',
 '天津': '天津市',
 '河北': '河北省',
 '山西': '山西省',
 '内蒙古': '内蒙古自治区',
 '辽宁': '辽宁省',
 '吉林': '吉林省',
 '黑龙江': '黑龙江省',
 '上海': '上海市',
 '江苏': '江苏省',
 '浙江': '浙江省',
 '安徽': '安徽省',
 '福建': '福建省',
 '江西': '江西省',
 '山东': '山东省',
 '河南': '河南省',
 '湖北': '湖北省',
 '湖南': '湖南省',
 '广东': '广东省',
 '广西': '广西壮族自治区',
 '海南': '海南省',
 '重庆': '重庆市',
 '四川': '四川省',
 '贵州': '贵州省',
 '云南': '云南省',
 '西藏': '西藏自治区',
 '陕西': '陕西省',
 '甘肃': '甘肃省',
 '青海': '青海省',
 '宁夏': '宁夏回族自治区',
 '新疆': '新疆维吾尔自治区',
'台湾':'台湾',
 '香港':'香港',
 '澳门':'澳门'}

In [39]:
df_tmp1 = df[['城市','销量']]
df_counts = df_tmp1.groupby('城市').sum()
m1 = (
        Map()
        .add('假期出行分布', [list(z) for z in zip([dictcode[x] for x in df_counts.index.values.tolist() ], df_counts.values.tolist())], 'china')
        .set_global_opts(
        title_opts=opts.TitleOpts(title='假期出行数据地图分布'),
        visualmap_opts=opts.VisualMapOpts(max_=100000, is_piecewise=False,range_color=["white", "#fa8072", "#ed1941"]),
        )
    )
m1.render_notebook()

# 华东、华南、华中等地区属于国民出游热点地区，尤其是北京、上海、江苏、广东、四川、陕西等地区出行比较密集。
# 各省市4A-5A景区数量柱状图

In [40]:
# 线性渐变
color_js = """new echarts.graphic.LinearGradient(0, 1, 0, 0,
    [{offset: 0, color: '#009ad6'}, {offset: 1, color: '#ed1941'}], false)""" 

df_tmp2 = df[df['星级'].isin(['4A', '5A'])]
df_counts = df_tmp2.groupby('城市').count()['星级']
b2 = (
        Bar()
            .add_xaxis(df_counts.index.values.tolist())
            .add_yaxis('4A-5A景区数量', df_counts.values.tolist(),itemstyle_opts=opts.ItemStyleOpts(color=JsCode(color_js)))
            .set_global_opts(
            title_opts=opts.TitleOpts(title='各省市4A-5A景区数量'),
            datazoom_opts=[opts.DataZoomOpts(), opts.DataZoomOpts(type_='inside')],
        )
    )
b2.render_notebook()

# 各省市4A-5A景区数量玫瑰图

In [41]:
df0 = df_counts.copy()
df0.sort_values(ascending=False, inplace=True)
c1 = (
    Pie()
    .add('', [list(z) for z in zip(df0.index.values.tolist(), df0.values.tolist())],
         radius=['30%', '100%'],
         center=['50%', '60%'],
         rosetype='area',
         )
    .set_global_opts(title_opts=opts.TitleOpts(title='地区景点数量'),
                     legend_opts=opts.LegendOpts(is_show=False),
                     toolbox_opts=opts.ToolboxOpts())
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True, position='inside', font_size=12,
                                               formatter='{b}: {c}', font_style='italic',
                                               font_weight='bold', font_family='Microsoft YaHei'
                                               ))
)
c1.render_notebook()

# 各省市4A-5A景区数量阴影散点图

In [42]:
item_style = {'normal': {'shadowColor': '#000000', 
                         'shadowBlur': 20,
                         'shadowOffsetX':5, 
                         'shadowOffsetY':15
                         }
              }
s1 = (
        Scatter()
        .add_xaxis(df_counts.index.values.tolist())
        .add_yaxis('4A-5A景区数量', df_counts.values.tolist(),symbol_size=50,itemstyle_opts=item_style)
        .set_global_opts(visualmap_opts=opts.VisualMapOpts(is_show=False, 
                                              type_='size',
                                              range_size=[5,50]))
)
s1.render_notebook()

# 各省市4A-5A景区地图分布

In [43]:
df_tmp3 = df[df['星级'].isin(['4A', '5A'])]
df_counts = df_tmp3.groupby('城市').count()['星级']
m2 = (
    Map()
    .add('4A-5A景区分布', [list(z) for z in zip([dictcode[x] for x in df_counts.index.values.tolist() ], df_counts.values.tolist())], 'china')
    .set_global_opts(
    title_opts=opts.TitleOpts(title='地图数据分布'),
    visualmap_opts=opts.VisualMapOpts(max_=50, is_piecewise=True),
    )
)
m2.render_notebook()

# 江苏、安徽、河南、北京、湖北等地区4A、5A级景区数量比较多。

# 3.7 门票价格区间占比玫瑰图

In [44]:
# 门票价格占比
price_level = [0, 50, 100, 150, 200, 250, 300, 350, 400, 500]    
label_level = ['0-50', '50-100', '100-150', '150-200', '200-250', '250-300', '300-350', '350-400', '400-500']    
jzmj_cut = pd.cut(df['价格'], price_level, labels=label_level)        
df_price = jzmj_cut.value_counts()
df_price

价格
0-50       888
50-100     725
100-150    278
150-200    184
200-250     62
250-300     59
300-350     20
350-400     19
400-500     13
Name: count, dtype: int64

In [45]:
p1 = (
    Pie(init_opts=opts.InitOpts(
            width='800px', height='600px',
            )
       )
        .add(
        '',
        [list(z) for z in zip(df_price.index.tolist(), df_price.values.tolist())],
        radius=['20%', '60%'],
        center=['40%', '50%'],
        rosetype='radius',
        label_opts=opts.LabelOpts(is_show=True),
        )    
        .set_global_opts(title_opts=opts.TitleOpts(title='门票价格占比',pos_left='33%',pos_top="5%"),
                        legend_opts=opts.LegendOpts(type_='scroll', pos_left="80%",pos_top="25%",orient="vertical")
                        )
        .set_series_opts(label_opts=opts.LabelOpts(formatter='{b}: {c} ({d}%)'),position='outside')
    )
p1.render_notebook()

# 3.8 门票价格区间数量散点图

In [46]:
color_js = """new echarts.graphic.RadialGradient(
                    0.5, 0.5, 1,
                    [{offset: 0,
                      color: '#009ad6'},
                     {offset: 1,
                      color: '#ed1941'}
                      ])"""

s2 = (
        Scatter()
        .add_xaxis(df_price.index.tolist())
        .add_yaxis('门票价格区间', df_price.values.tolist(),symbol_size=50,itemstyle_opts=opts.ItemStyleOpts(color=JsCode(color_js))) 
        .set_global_opts(
            yaxis_opts=opts.AxisOpts(name='数量'),
            xaxis_opts=opts.AxisOpts(name='价格区间(元)'))
        .set_global_opts(visualmap_opts=opts.VisualMapOpts(is_show=False, 
                                              # 设置通过图形大小来表现数据
                                              type_='size',
                                              # 图形大小映射范围
                                              range_size=[5,50]))
)
s2.render_notebook()

# 3.9 景点简介词云

In [47]:
contents = "".join('%s' % i for i in df['简介'].values.tolist())
contents_list = jieba.cut(contents)
ac = Counter(contents_list)

# stopwords = []
# with open('./stopwords.txt', "r",encoding='utf-8') as f:  # 打开文件
#     data = f.read()  # 读取文件
#     stopwords = data.split('\n')

# for i in stopwords:
#     del ac[i]
 
w1 = (
    WordCloud()
    .add("", 
         ac.most_common(150), 
         word_size_range=[5, 100], 
         textstyle_opts=opts.TextStyleOpts(font_family="cursive"),
         shape='star')
    .set_global_opts(title_opts=opts.TitleOpts(title="景点简介词云"))
)
w1.render_notebook()